# nb15 — Tier 2 Full-Scale HDC-RWKV (Architecture Validation)

### What this notebook is for

Wozformer's hardware artifact (nb12c, 16.4 KB) produces fragmented output. A reviewer would reasonably ask:
**"Is the HDC-RWKV architecture fundamentally limited, or is the fragmentation just hardware-bound?"**

This notebook answers that by training the same architecture at *research scale* — no 32 KB EEPROM constraint, no
6502 cycle budget — and observing whether the output becomes meaningful.

### Three possible outcomes

| Outcome | Implication for paper |
|---|---|
| BPC < 2.5, real phrases appear | Architecture validated. nb12c framed as hardware ceiling, not architectural failure. |
| BPC 2.5–2.9, mixed quality | Architecture has limitations. Honest framing: 'binary recurrence trades quality for deployment.' |
| BPC > 2.9, still fragmented | Architecture has a hard ceiling at any size. Negative result of its own. |

### Configuration

- vocab=512 BPE, d=1024, L=2, NLL-only (no distillation, per O1)
- ~128 KB binary storage (would fit ESP32 flash with paging if we wanted to ship it)
- 30K training steps

### Why we changed from the proposed vocab=1024, d=1024

Per F5 (HDC capacity ceiling at d/log(d)): at d=1024 the capacity is ~100 prototypes. vocab=1024 would be 10× over.
vocab=512 with d=1024 is 5× over — still beyond capacity but observable. If we still see fragments, the
architecture's binary geometry is the cause, confirmed across multiple scales.


## Cell 1 — Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.chdir('/kaggle/working')
REPO_URL = 'https://github.com/elixpo/wozformer.git'
subprocess.run(['rm', '-rf', '/kaggle/working/wozformer'], check=True)
subprocess.run(['git', 'clone', REPO_URL, '/kaggle/working/wozformer'], check=True)
WOZFORMER_PATH = Path('/kaggle/working/wozformer')
os.chdir(WOZFORMER_PATH)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

# Force-evict any cached wozformer modules
for m in list(sys.modules):
    if m.startswith('wozformer'):
        del sys.modules[m]

import wozformer as wz
import torch
import math
import matplotlib.pyplot as plt
print(f'wozformer {wz.__version__}, device: {wz.utils.get_device()}')


## Cell 2 — Hyperparameters

Sized to demonstrate the architecture at research scale, not constrained by hardware.


In [ ]:
# Tier 2 architecture validation: vocab=512 BPE, d=1024, L=2
VOCAB_SIZE = 512
D          = 1024
N_LAYERS   = 2
BLOCK_SIZE = 64

# Training
BATCH_SIZE = 16     # smaller batch — d=1024 recurrence is memory-heavy
LR         = 3e-3
N_STEPS    = 30000  # twice nb14's budget — bigger model needs more steps
EVAL_EVERY = 1000
SEED       = 1337

RUN_DIR = Path('/kaggle/working/runs')
RUN_DIR.mkdir(parents=True, exist_ok=True)
TIER2_PT  = RUN_DIR / 'tier2_hdcrwkv.pt'
TIER2_BIN = RUN_DIR / 'tier2_hdcrwkv.bin'
BPE_JSON  = RUN_DIR / 'bpe_512.json'


## Cell 3 — Load corpus and train BPE


In [ ]:
wz.utils.set_seed(SEED)
device = wz.utils.get_device()

text = wz.data.load_corpus(WOZFORMER_PATH / 'data' / 'tinyshakespeare.txt')
tok = wz.tokenizer.BPETokenizer.train(text, vocab_size=VOCAB_SIZE)
tok.save(BPE_JSON)

ids = torch.tensor(tok.encode(text), dtype=torch.long)
train_data, val_data = wz.data.split_train_val(ids)
print(f'tokens: train {len(train_data):,} / val {len(val_data):,}')

# Show the BPE learned
long_toks = sorted([t for t in tok.itos if t.startswith('<') is False],
                   key=lambda x: -len(x))[:20]
print(f'longest 20 tokens: {long_toks}')


## Cell 4 — Build the full-scale student


In [ ]:
scfg = wz.config.HDCRWKVConfig(
    vocab_size=VOCAB_SIZE,
    d=D,
    n_layers=N_LAYERS,
    block_size=BLOCK_SIZE,
)
model = wz.models.HDCRWKV(scfg).to(device)

n_train = wz.utils.count_params(model)
n_bytes = model.deployment_bytes()
print(f'trainable params: {n_train:,}')
print(f'deployment bytes (binary): {n_bytes:,} ({n_bytes/1024:.1f} KB)')

# Sanity: forward should produce loss near ln(VOCAB)
xb, yb = wz.data.make_batch(train_data, BATCH_SIZE, BLOCK_SIZE, device)
with torch.no_grad():
    _, loss = model(xb, yb)
print(f'init loss: {loss.item():.4f} (expect ~{math.log(VOCAB_SIZE):.4f} = ln(V))')


## Cell 5 — Train NLL-only (no distillation per O1)

~30K steps. ~2-3 hours on T4. The trainer saves the best-hard-val checkpoint.

### What to watch

- val(hard) should descend toward ~3.0–3.5 nats over the first 10K steps
- If it plateaus by 15K, the architecture has hit its ceiling at this configuration
- If it keeps descending, we still have room and could train longer


In [ ]:
train_cfg = wz.config.TrainConfig(
    batch_size=BATCH_SIZE,
    block_size=BLOCK_SIZE,
    lr=LR,
    n_steps=N_STEPS,
    eval_every=EVAL_EVERY,
    seed=SEED,
    weight_decay=0.0,
)
history, best = wz.trainer.train(
    model, train_data, val_data, train_cfg,
    device=device, eval_hard=True,
)
print(f'\nbest HARD val: {best["val"]:.4f} at step {best["step"]}')


## Cell 6 — Plot loss curves and report BPC


In [ ]:
steps = [h[0] for h in history]
trains = [h[1] for h in history]
vals_soft = [h[2] for h in history]
vals_hard = [h[3] for h in history]

plt.figure(figsize=(10, 4.5))
plt.plot(steps, trains, label='train', alpha=0.6)
plt.plot(steps, vals_soft, label='val(soft)', alpha=0.8)
plt.plot(steps, vals_hard, label='val(hard, deployment)', linewidth=2, color='C2')
plt.axhline(math.log(VOCAB_SIZE), color='gray', linestyle=':', label=f'random (ln {VOCAB_SIZE})')
# Reference: nb12c BPC 2.93 at vocab=256 (different vocab so just for context)
plt.xlabel('step'); plt.ylabel('cross-entropy (nats/token)')
plt.title(f'Tier 2 HDC-RWKV (d={D}, V={VOCAB_SIZE}, L={N_LAYERS}) — architecture at scale')
plt.legend(); plt.grid(alpha=0.3); plt.show()

# BPC
sample = val_data[:5000].tolist()
avg_cpt = wz.metrics.avg_chars_per_token(tok, sample)
bpc = wz.metrics.bits_per_char(best['val'], avg_cpt)
print(f'avg chars/token: {avg_cpt:.2f}')
print(f'best HARD val:   {best["val"]:.4f} nats/token')
print(f'Tier 2 BPC:      {bpc:.4f}')
print()
print('Reference points:')
print(f'  Teacher (vocab=256, d=192 fp32):  BPC 2.04')
print(f'  nb12c HDC-RWKV (vocab=256, d=256, 16KB): BPC 2.93')
if bpc < 2.5:
    print('\n  >> Tier 2 architecture validation: PASSED (BPC < 2.5)')
elif bpc < 2.9:
    print('\n  >> Mixed quality (BPC 2.5–2.9). Architecture has limitations.')
else:
    print('\n  >> Hard ceiling confirmed (BPC > 2.9). Architecture vs scale: independent.')


## Cell 7 — Generate samples and judge by eye

This is the qualitative test. Compare to:

- Teacher output: real Shakespeare characters, full sentences, dialogue format
- nb12c output: real words mixed with fragments, no sustained sentences

What we want to see in Tier 2: multi-word real phrases, recognizable sentence shape,
correct character names. If we see those → architecture validated.


In [ ]:
prompts = ['king', 'romeo', 'my lord,', 'queen elizabeth:', 'to be or not']
seeds = [1337, 42, 7, 99, 2024]

for prompt, seed in zip(prompts, seeds):
    print(f'\n===== {prompt!r}  seed={seed} =====')
    out = wz.generate.generate(
        model, tok, prompt=prompt, max_new_tokens=120,
        block_size=BLOCK_SIZE, temperature=0.7, top_k=10,
        seed=seed, device=device, use_hard=True,
    )
    print(out)


## Cell 8 — Save Tier 2 checkpoint + binary


In [ ]:
import struct
import numpy as np

torch.save(
    {
        'config': {
            'vocab_size': VOCAB_SIZE, 'd': D, 'n_layers': N_LAYERS,
            'block_size': BLOCK_SIZE,
        },
        'model_state': model.state_dict(),
        'history': history,
        'best_val_hard': best['val'],
        'best_step': best['step'],
        'n_train_params': n_train,
        'deploy_bytes': n_bytes,
        'tier': 'tier2_architecture_validation',
    },
    TIER2_PT,
)
print(f'saved Tier 2 .pt → {TIER2_PT}  ({TIER2_PT.stat().st_size/1024:.1f} KB)')

# Bit-pack into deployment binary (matches v3 format)
def pack_bits(t):
    return np.packbits((t > 0).to(torch.uint8).cpu().numpy(), axis=-1, bitorder='big')

vocab_packed = pack_bits(model.vocab_hv_c.data)
proto_packed = pack_bits(model.prototype_hv_c.data)
decay_packed = [pack_bits(dm.data.unsqueeze(0)).squeeze(0) for dm in model.decay_masks_c]

buf = bytearray()
buf += b'WHR3'
buf += bytes([3])                                  # version
buf += struct.pack('<H', VOCAB_SIZE)
buf += struct.pack('<H', D // 8)
buf += bytes([BLOCK_SIZE, N_LAYERS])
buf += struct.pack('<f', model.log_temp.item())
buf += bytes(3)                                    # reserved
buf += vocab_packed.tobytes()
buf += proto_packed.tobytes()
for dp in decay_packed:
    buf += dp.tobytes()

TIER2_BIN.write_bytes(buf)
print(f'saved Tier 2 .bin → {TIER2_BIN}  ({len(buf):,} bytes = {len(buf)/1024:.1f} KB)')


## Post-mortem — what we learned

After running, update [findings.md](../docs/findings.md) with:

- Final BPC + qualitative judgment of generation samples
- New finding if BPC < 2.5 (architecture validated at scale)
- New finding if BPC > 2.9 (architecture has scale-independent ceiling)
- Reference for the paper's Tier 2 contribution

Download the artifacts from `/kaggle/working/runs/`:

- `tier2_hdcrwkv.pt` — Python checkpoint for evaluation
- `tier2_hdcrwkv.bin` — deployable binary (~128 KB, ESP32 fit)
- `bpe_512.json` — tokenizer used
